In [ ]:
# -----------------------------
# Mumbai House Price Prediction
# -----------------------------
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import pickle

# -----------------------------
# PART 1: Load Data
# -----------------------------
file_path = r"C:\Users\Parth Koli\Downloads\mumbai_house_price.csv"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"Boss, file not found: {file_path}")

df = pd.read_csv(file_path)

# -----------------------------
# PART 2: Clean Data
# -----------------------------
# Strip and unify locality names (remove extra spaces, lowercase)
df['locality'] = df['locality'].astype(str).str.strip().str.title()

# Manual mapping for known variations
locality_corrections = {
    'Dombivli': 'Dombivali',
    'Dombivli W': 'Dombivali',
    'Vile Parle East': 'Vile Parle',
    'Nallasopara W': 'Nalasopara',
    'Nalasopara': 'Nalasopara'
}
df['locality'] = df['locality'].replace(locality_corrections)

# Remove unrealistic prices, price_per_sqft, area
df = df[(df['price'] > 1000000) & (df['price'] < 100000000)]  # 10 lakh -> 10 crore
df = df[(df['price_per_sqft'] > 2000) & (df['price_per_sqft'] < 40000)]
df = df[(df['area'] > 250) & (df['area'] < 5000)]

# Fill missing features with median
features = ['area', 'bedroom_num', 'bathroom_num', 'balcony_num', 'age', 'total_floors', 'price_per_sqft']
for col in features:
    df[col].fillna(df[col].median(), inplace=True)

df = df.dropna(subset=['price'])

# -----------------------------
# PART 3: Features & Target
# -----------------------------
X = df[features].values
y = df['price'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# PART 4: Train Model
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Random Forest Regressor (more robust than Linear Regression)
model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
model.fit(X_train, y_train)

# Test accuracy
y_pred = model.predict(X_test)
accuracy = r2_score(y_test, y_pred)
print(f"Boss, Random Forest R² accuracy: {accuracy:.4f}")

# Save model and scaler
pickle.dump(model, open("mumbai_price_model.pkl", "wb"))
pickle.dump(scaler, open("mumbai_scaler.pkl", "wb"))

# -----------------------------
# PART 5: Optional: Test Predictions
# -----------------------------
new_property = np.array([[1200, 3, 2, 1, 5, 10, 12000]])  # area, bedrooms, bathrooms, balcony, age, floors, price_per_sqft
new_property_scaled = scaler.transform(new_property)
predicted_price = model.predict(new_property_scaled)[0]
print(f"🏠 Predicted Price: ₹{int(predicted_price)}")

C:\Users\Parth Koli\AppData\Local\Temp\ipykernel_9948\2498747517.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\Parth Koli\AppData\Local\Temp\ipykernel_9948\2498747517.py:46: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


Boss, Random Forest R² accuracy: 0.9995
🏠 Predicted Price: ₹14270290


In [ ]:
import pickle

# After training your model
model_filename = "mumbai_price_model.pkl"
scaler_filename = "mumbai_scaler.pkl"

# Save the model and scaler
with open(model_filename, "wb") as f:
    pickle.dump(model, f)

with open(scaler_filename, "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved!")

Model and scaler saved!
